In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')

    # Update this path to your zip file location in Drive:
    zip_file_path = '/content/drive/MyDrive/DATASCI266/Final Project Notebooks/Juliet_Test_Suite_v1.3_for_C_Cpp.zip'
    print(f'Using zip file path: {zip_file_path}')

except Exception as e:
    print('Not running in Google Colab, or Google Drive mount failed.')
    print('If you are running locally, please set zip_file_path manually.')
    print('Error:', e)


Not running in Google Colab, or Google Drive mount failed.
If you are running locally, please set zip_file_path manually.
Error: No module named 'google.colab'


In [1]:
import os
import zipfile

# Path to your local zip file
zip_file_path = os.path.expanduser('C:/Users/Jennifer_Nishimura/Documents/DATASCI266/final_project/W266-Final-Project/Juliet_Test_Suite_v1.3_for_C_Cpp.zip')

# Check if file exists
if not os.path.exists(zip_file_path):
    print(f"Error: Zip file not found at {zip_file_path}")
    print("Please provide the path to your zip file:")
    print(f"  - Place it in: {zip_file_path}")
    print(f"  - Or update the zip_file_path variable above")
    raise FileNotFoundError(f"Zip file not found: {zip_file_path}")

print(f"Found zip file: {zip_file_path}")


Found zip file: C:/Users/Jennifer_Nishimura/Documents/DATASCI266/final_project/W266-Final-Project/Juliet_Test_Suite_v1.3_for_C_Cpp.zip


### EDA

In [2]:
import zipfile
import os
import pandas as pd
import random

# Read directly from the zip file and sample Juliet C/C++ files
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    all_source_files = [f for f in zip_ref.namelist() if f.endswith(('.c', '.cpp'))]

print(f"Total source files in zip: {len(all_source_files)}")

# Keep only Juliet test cases that contain vulnerable or safe variants
def is_juliet_variant(fname):
    lower = fname.lower()
    return 'bad' in lower or 'goodg2b' in lower or 'goodb2g' in lower

variant_files = [f for f in all_source_files if is_juliet_variant(os.path.basename(f))]
print(f"Juliet vulnerable/safe variant files: {len(variant_files)}")

if not variant_files:
    raise ValueError('No Juliet vulnerable/safe variant files found in the zip archive.')

# Create a DataFrame of filtered variant files
variant_df = pd.DataFrame([
    {
        'zip_path': f,
        'filename': os.path.basename(f),
        'label': 'vulnerable' if 'bad' in os.path.basename(f).lower() else 'safe',
        'cwe_category': next((segment for segment in f.split('/') if segment.startswith('CWE')), 'Unknown')
    }
    for f in variant_files
])

# Sample from the filtered variant DataFrame
sample_size = min(2000, len(variant_df))
sample_df = variant_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f"Sampled {sample_size} Juliet variant files")

# Read sampled file contents from the zip archive
sample_df['code'] = ''
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    for idx, row in sample_df.iterrows():
        with zip_ref.open(row['zip_path']) as f:
            sample_df.at[idx, 'code'] = f.read().decode('utf-8', errors='ignore')

print(sample_df['label'].value_counts())
display(sample_df.head())


Total source files in zip: 101235
Juliet vulnerable/safe variant files: 10191
Sampled 2000 Juliet variant files
label
safe          1172
vulnerable     828
Name: count, dtype: int64


,zip_path,filename,label,cwe_category,code
0,C/testcases/CWE124_Buffer_Underwrite/s04/CWE12...,CWE124_Buffer_Underwrite__wchar_t_declare_memm...,vulnerable,CWE124_Buffer_Underwrite,/* TEMPLATE GENERATED TESTCASE FILE\r\nFilenam...
1,C/testcases/CWE134_Uncontrolled_Format_String/...,CWE134_Uncontrolled_Format_String__wchar_t_fil...,safe,CWE134_Uncontrolled_Format_String,/* TEMPLATE GENERATED TESTCASE FILE\r\nFilenam...
2,C/testcases/CWE134_Uncontrolled_Format_String/...,CWE134_Uncontrolled_Format_String__char_listen...,vulnerable,CWE134_Uncontrolled_Format_String,/* TEMPLATE GENERATED TESTCASE FILE\r\nFilenam...
3,C/testcases/CWE134_Uncontrolled_Format_String/...,CWE134_Uncontrolled_Format_String__char_file_v...,safe,CWE134_Uncontrolled_Format_String,/* TEMPLATE GENERATED TESTCASE FILE\r\nFilenam...
4,C/testcases/CWE122_Heap_Based_Buffer_Overflow/...,CWE122_Heap_Based_Buffer_Overflow__c_CWE129_ra...,vulnerable,CWE122_Heap_Based_Buffer_Overflow,/* TEMPLATE GENERATED TESTCASE FILE\r\nFilenam...


### EDA

### Creating a Sample Set

In [3]:
def make_prompts(code):
    prompt_explain = (
        "Explain any vulnerabilities in the following C code. "
        "Be specific and describe the root cause:\n\n"
        f"{code}\n"
    )

    prompt_fix = (
        "Rewrite the following C code to be secure. "
        "Use safe APIs and best practices:\n\n"
        f"{code}\n"
    )

    return prompt_explain, prompt_fix


In [4]:
def generate_outputs(model, tokenizer, code):
    prompt_explain, prompt_fix = make_prompts(code)

    # Explanation
    inputs_exp = tokenizer(prompt_explain, return_tensors="pt").to(model.device)
    exp_output = model.generate(**inputs_exp, max_new_tokens=256)
    explanation = tokenizer.decode(exp_output[0], skip_special_tokens=True)

    # Secure rewrite
    inputs_fix = tokenizer(prompt_fix, return_tensors="pt").to(model.device)
    fix_output = model.generate(**inputs_fix, max_new_tokens=256)
    fixed_code = tokenizer.decode(fix_output[0], skip_special_tokens=True)

    return explanation, fixed_code


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

c:\Users\Jennifer_Nishimura\Documents\DATASCI266\final_project\W266-Final-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9264.24it/s]


In [6]:

# sample_df["explanation"] = ""
# sample_df["generated_code"] = ""

# for idx, row in sample_df.iterrows():
#     print(idx)
#     code = row["code"]
#     explanation, fixed_code = generate_outputs(model, tokenizer, code)

#     sample_df.at[idx, "explanation"] = explanation
#     sample_df.at[idx, "generated_code"] = fixed_code


In [7]:
# In Parallel 
from joblib import Parallel, delayed
from tqdm import tqdm
# Wrap your generation function so workers only receive the row
def generate_for_row(row):
    code = row["code"]
    explanation, fixed_code = generate_outputs(model, tokenizer, code)
    return explanation, fixed_code

# Run parallel inference

results = Parallel(n_jobs=6)(
    delayed(generate_for_row)(row)
    for row in tqdm(sample_df.to_dict("records"))
)


# Insert results back into the DataFrame
sample_df["explanation"] = [r[0] for r in results]
sample_df["generated_code"] = [r[1] for r in results]


  2%|▏         | 48/2000 [15:32<11:42:10, 21.58s/it]c:\Users\Jennifer_Nishimura\Documents\DATASCI266\final_project\W266-Final-Project\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 2000/2000 [13:34:15<00:00, 24.43s/it]  


In [8]:
sample_df.to_csv("juliet_sample_with_explanations_and_fixes.csv", index=False)

In [10]:
print(sample_df["explanation"][1])
print(sample_df["generated_code"][1])

Explain any vulnerabilities in the following C code. Be specific and describe the root cause:

/* TEMPLATE GENERATED TESTCASE FILE
Filename: CWE134_Uncontrolled_Format_String__wchar_t_file_vprintf_81_goodG2B.cpp
Label Definition File: CWE134_Uncontrolled_Format_String.vasinks.label.xml
Template File: sources-vasinks-81_goodG2B.tmpl.cpp
*/
/*
 * @description
 * CWE: 134 Uncontrolled Format String
 * BadSource: file Read input from a file
 * GoodSource: Copy a fixed string into data
 * Sinks: vprintf
 *    GoodSink: vwprintf with a format string
 *    BadSink : vwprintf without a format string
 * Flow Variant: 81 Data flow: data passed in a parameter to an virtual method called via a reference
 *
 * */
#ifndef OMITGOOD

#include <stdarg.h>
#include "std_testcase.h"
#include "CWE134_Uncontrolled_Format_String__wchar_t_file_vprintf_81.h"

namespace CWE134_Uncontrolled_Format_String__wchar_t_file_vprintf_81
{

static void goodG2BVaSink(wchar_t * data, ...)
{
    {
        va_list args;
    

# Marking Hallucinations ~ Rule Based Hallucination Checking

In [9]:
import re

def extract_functions(code):
    return set(re.findall(r'\b([A-Za-z_]\w*)\s*\(', code))

def extract_headers(code):
    return set(re.findall(r'#include\s*<([^>]+)>', code))

def extract_identifiers(code):
    return set(re.findall(r'\b[A-Za-z_]\w*\b', code))


In [ ]:
juliet_funcs = extract_functions(sample_df["code"].iloc[0])
juliet_ids = extract_identifiers(sample_df["code"].iloc[0])


In [ ]:
VALID_APIS = {
    "memmove", "memcpy", "wmemset", "printf", "vwprintf",
    "malloc", "free", "strlen", "strcpy", "strncpy",
    "printLine", "printWLine", "action"
}


In [ ]:
def has_invented_api(model_code):
    funcs = extract_functions(model_code)
    return any(f not in VALID_APIS for f in funcs)


In [ ]:
VALID_HEADERS = {"stdio.h", "stdlib.h", "string.h", "wchar.h"}

def has_invented_header(model_code):
    headers = extract_headers(model_code)
    return any(h not in VALID_HEADERS for h in headers)


In [ ]:
INVENTED_VULN_KEYWORDS = {
    "sql injection", "remote code execution", "xss",
    "race condition", "deadlock"
}

def has_invented_vulnerability(text):
    return any(k in text.lower() for k in INVENTED_VULN_KEYWORDS)


In [ ]:
def has_invented_identifiers(juliet_code, model_code):
    juliet_ids = extract_identifiers(juliet_code)
    model_ids = extract_identifiers(model_code)
    invented = model_ids - juliet_ids
    invented -= {"int", "char", "return", "void", "const"}
    return len(invented) > 0


In [ ]:
def label_hallucination(juliet_code, explanation, generated_code):
    if has_invented_api(generated_code): return 1
    if has_invented_header(generated_code): return 1
    if has_invented_vulnerability(explanation): return 1
    if has_invented_identifiers(juliet_code, generated_code): return 1
    return 0


In [ ]:
sample_df["hallucinated"] = [
    label_hallucination(row["code"], row["explanation"], row["generated_code"])
    for _, row in sample_df.iterrows()
]
